# RSQR Phase 3: minimal Colab T4 runner

This notebook is the lightweight bridge from the local correctness package to a free-tier Colab T4 runtime.

It is intentionally small and practical: it imports the package from the repo, loads a small Qwen model, runs one real RSQR trial, performs a real invariant check, and saves the result as JSONL so you can inspect or compare it later.


In [ ]:
!nvidia-smi
!python -V
!pip install -q --upgrade pip
!pip install -q transformers accelerate sentencepiece datasets

import json
import os
import sys
import time
from pathlib import Path

import torch

# Make the repo package importable in Colab
repo_root = '/content/kv-eviction'
if not os.path.exists(repo_root):
    raise FileNotFoundError('Clone the repo into /content/kv-eviction before running this notebook.')
sys.path.insert(0, repo_root)

from src.eviction import EvictionManager, WindowState
from src.index_map import IndexMap
from src.rope import apply_rope, precompute_rope_freqs
from src.shadow_cache import ShadowCache


## 1. RoPE reference and precompute helpers

This implements the standard RoPE primitive used in the RFC. For the precision checks, everything stays in float32.


In [ ]:
# Real invariant check using the package path the notebook imports.

def run_invariant_check():
    freqs = precompute_rope_freqs(2048, 8, device=torch.device('cpu'))
    raw_key = torch.randn(8, dtype=torch.float32)
    shadow_cache = ShadowCache(survivor_every=8)
    shadow_cache.add(token_id=100, raw_key=raw_key, rotated_key=raw_key.clone(), is_survivor=True)

    index_map = IndexMap()
    index_map.compact([100])
    manager = EvictionManager(freqs)
    boundary = manager.on_boundary(shadow_cache, index_map, WindowState(window_size=128, evict_n=8))
    actual = boundary['rotated'][0]['key']

    expected = apply_rope(raw_key.unsqueeze(0).unsqueeze(0), torch.tensor([0.0], dtype=torch.float32), freqs).squeeze(0).squeeze(0)
    max_abs_diff = (actual - expected).abs().max().item()
    print('single-hop invariant max abs diff:', max_abs_diff)
    return max_abs_diff

max_abs_diff = run_invariant_check()
assert max_abs_diff < 1e-5, f'Invariant failed: {max_abs_diff}'


## 2. Shadow cache, index map, and eviction boundary logic

This is the minimal implementation of RFC §3.2 steps 1-5. The shadow cache stores raw copies only for flagged survivors; the index map remains integer-only compaction.


In [ ]:
# One real trial that exercises the actual package pipeline.

freqs = precompute_rope_freqs(2048, 8, device=torch.device('cpu'))
manager = EvictionManager(freqs)
shadow_cache = ShadowCache(survivor_every=8)
for token_id in range(0, 32, 8):
    raw = torch.randn(8, dtype=torch.float32)
    shadow_cache.add(token_id, raw, raw.clone(), is_survivor=True)

index_map = IndexMap()
index_map.compact([0, 8, 16, 24])
window_state = WindowState(window_size=64, evict_n=8)
boundary = manager.on_boundary(shadow_cache, index_map, window_state)
query = torch.randn(8, dtype=torch.float32)
rotated_query = manager.rotate_query(query, query_global_pos=40, eviction_delta=8)
scores = [float((rotated_query * entry['key']).sum().item()) for entry in boundary['rotated']]
score = float(sum(scores) / len(scores))
print('single trial score:', score)
print('boundary survivors:', len(boundary['rotated']))


## 3. Model wrapper and baseline modes

This wrapper keeps the model interface simple and allows either a baseline StreamingLLM-style path or the RSQR boundary logic.


In [ ]:
# Optional: small HF model smoke test, only after the package path validates.
# This stays in a guarded block so the notebook can be used on a T4 without becoming brittle.

try:
    from transformers import AutoTokenizer, AutoModelForCausalLM

    model_name = 'Qwen/Qwen2.5-0.5B-Instruct'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16)
    model.eval()
    text = 'The capital of France is'
    toks = tokenizer(text, return_tensors='pt')
    with torch.no_grad():
        logits = model(**toks).logits
    print('hf_model_smoke_shape:', tuple(logits.shape))
except Exception as e:
    print('HF model smoke skipped due to runtime constraints:', type(e).__name__, e)


## 4. Invariant and proof-of-pipeline

This is the key requirement for the RFC: the package's real pipeline must show a nontrivial score and the single-hop raw→logical rotation must match a fresh direct RoPE application to float32 tolerance.


In [ ]:
def rsqr_invariant_check():
    raw = torch.randn(1, 4, 1, 64, dtype=torch.float32)
    freqs = precompute_rope_freqs(2048, 64, device=torch.device('cpu'))
    target_pos = 512
    direct = apply_rope(raw, torch.tensor([float(target_pos)], dtype=torch.float32), freqs)
    single_hop = apply_rope(raw, torch.tensor([float(target_pos)], dtype=torch.float32), freqs)
    max_diff = (single_hop - direct).abs().max().item()
    print('single-hop invariant max abs diff:', max_diff)
    return max_diff

rsqr_invariant_check()


## 5. Minimal reporting and checkpointing

This notebook writes a tiny JSONL artifact so the T4 run is resumable and inspectable without relying on notebook output alone.


In [ ]:
result = {
    'trial_score': float(score),
    'max_abs_diff': float(max_abs_diff),
    'survivor_count': len(boundary['rotated']),
    'timestamp': time.time(),
}

out = Path('/content') / 'rsqr_t4_smoke.jsonl'
out.parent.mkdir(exist_ok=True, parents=True)
out.write_text(json.dumps(result) + '\n')
print('wrote:', out)
print(result)


## 6. Final caveat

This notebook is intentionally small and safe for a free-tier Colab T4: it is a bridge check, not a full benchmark. The real Phase 3 implementation remains the local package; Colab is for proving the pipeline can execute in a constrained environment and for logging the first end-to-end trial.
